# Multi-Float Simulation Error Analysis

Discovers Argo floats that operated inside the CMEMS model domain, downloads their trajectory
files, reruns the simulation against each one using the **reset-per-cycle** approach, and
aggregates the position errors across all floats.

**Workflow:**
1. **Cell 1** — reads `manifest.yaml`, queries the Argo index, shows a map of floats in the domain.
2. **Cell 2** — edit `WMOS_TO_ANALYZE` with WMOs from the map.
3. **Cell 3** — downloads `{WMO}_Rtraj.nc` from the Argo GDAC for each selected float.
4. **Cells 4–7** — setup, helpers, simulation loop, aggregate plots.

Run all: `Kernel › Restart & Run All`

In [1]:
# ── Cell 1: Discover floats in the CMEMS domain ──────────────────────────────
import sys
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import plotly.graph_objects as go

sys.path.insert(0, str(Path("..") / "src"))
from data_loader import load_manifest

DATA_DIR      = Path("..") / "data" / "raw"
ARGO_DATA_DIR = Path("..") / "data" / "argo_data"

# ── Derive CMEMS domain from manifest ────────────────────────────────────────
manifest = load_manifest(DATA_DIR)

lat_min = min(e["lat"][0] for e in manifest)
lat_max = max(e["lat"][1] for e in manifest)
lon_min = min(e["lon"][0] for e in manifest)
lon_max = max(e["lon"][1] for e in manifest)
t_start = min(datetime.fromisoformat(e["time"][0]) for e in manifest)
t_end   = max(datetime.fromisoformat(e["time"][1]) for e in manifest)

print(f"CMEMS domain : lat=[{lat_min:.2f}, {lat_max:.2f}]  lon=[{lon_min:.2f}, {lon_max:.2f}]")
print(f"Time window  : {t_start.date()} → {t_end.date()}")

# ── Load Argo global index and filter to CMEMS domain ────────────────────────
from argopy import ArgoIndex

print("\nLoading Argo global index...")
df_all = ArgoIndex().load().to_dataframe()  # wmo, dac, latitude, longitude, date columns

mask = (
    (df_all["latitude"]  >= lat_min) & (df_all["latitude"]  <= lat_max) &
    (df_all["longitude"] >= lon_min) & (df_all["longitude"] <= lon_max) &
    (df_all["date"] >= pd.Timestamp(t_start)) &
    (df_all["date"] <= pd.Timestamp(t_end))
)
_df = df_all[mask].copy()

# WMO → DAC mapping used by the download cell
_wmo_to_dac = _df.drop_duplicates("wmo").set_index("wmo")["dac"].to_dict()

summary = (
    _df.groupby("wmo")
    .agg(n_profiles=("date", "count"),
         first_profile=("date", "min"),
         last_profile=("date", "max"))
    .sort_values("n_profiles", ascending=False)
    .reset_index()
)

print(f"\nFound {len(summary)} floats with {len(_df)} total profiles:")
print(summary.to_string(index=False))

# ── Map ───────────────────────────────────────────────────────────────────────
fig = go.Figure()
fig.add_trace(go.Scattergeo(
    lon=[lon_min, lon_max, lon_max, lon_min, lon_min],
    lat=[lat_min, lat_min, lat_max, lat_max, lat_min],
    mode="lines", line=dict(color="cyan", width=2, dash="dash"),
    name="CMEMS domain",
))
for wmo in summary["wmo"]:
    pts = _df[_df["wmo"] == wmo].sort_values("date")
    fig.add_trace(go.Scattergeo(
        lon=pts["longitude"], lat=pts["latitude"],
        mode="lines+markers", marker=dict(size=3), line=dict(width=1),
        name=str(wmo),
        hovertemplate=f"WMO {wmo}<br>%{{lat:.3f}}°N %{{lon:.3f}}°E<extra></extra>",
    ))
fig.update_layout(
    title=f"Argo floats in CMEMS domain — {len(summary)} floats",
    geo=dict(
        scope="europe",
        center=dict(lat=(lat_min + lat_max) / 2, lon=(lon_min + lon_max) / 2),
        projection_scale=6,
        showland=True,  landcolor="rgb(40,40,40)",
        showocean=True, oceancolor="rgb(10,30,60)",
        showcoastlines=True, coastlinecolor="grey", showframe=False,
    ),
    paper_bgcolor="rgb(15,20,40)", font_color="white", height=600,
    legend=dict(title="WMO", bgcolor="rgba(0,0,0,0.5)"),
)
fig.show()

CMEMS domain : lat=[53.50, 56.50]  lon=[12.50, 18.00]
Time window  : 2023-10-01 → 2025-01-31


/home/ddyob/Documents/argo_piloting/sim/.venv/bin/python: No module named pip



Loading Argo global index...

Found 6 floats with 1068 total profiles:
    wmo  n_profiles       first_profile        last_profile
4903784         317 2023-11-07 20:40:05 2025-01-27 11:50:38
3902117         285 2023-10-01 08:14:30 2024-02-24 06:20:30
2903902         175 2024-03-22 18:11:31 2025-01-26 13:43:28
1902682         154 2023-10-02 09:28:30 2024-02-23 19:25:00
7902194         120 2024-05-28 12:02:00 2025-01-31 19:31:30
7900584          17 2023-10-03 10:29:10 2023-12-22 10:38:51


In [6]:
# ── Cell 2: Select floats to analyse ─────────────────────────────────────────
# Add WMO numbers from the map above.
WMOS_TO_ANALYZE = [
    7902194,
    4903784,
    1902682,
    7900584,
    3902117
]

In [7]:
# ── Cell 3: Download Rtraj files from Argo GDAC ───────────────────────────────
import urllib.request

# Use WMO→DAC mapping from Cell 1; re-query if this cell runs standalone.
try:
    wmo_to_dac = _wmo_to_dac
except NameError:
    print("Re-loading Argo index for DAC info...")
    import pandas as pd
    from argopy import ArgoIndex
    _fb = ArgoIndex().load().to_dataframe()
    wmo_to_dac = _fb.drop_duplicates("wmo").set_index("wmo")["dac"].to_dict()

for wmo in WMOS_TO_ANALYZE:
    out_dir  = ARGO_DATA_DIR / f"argo_{wmo}"
    out_path = out_dir / f"{wmo}_Rtraj.nc"

    if out_path.exists():
        print(f"WMO {wmo}: Rtraj already present — skipping.")
        continue

    dac = wmo_to_dac.get(wmo)
    if dac is None:
        print(f"WMO {wmo}: DAC not found in index — skipping.")
        continue

    url = f"https://data-argo.ifremer.fr/dac/{dac}/{wmo}/{wmo}_Rtraj.nc"
    out_dir.mkdir(parents=True, exist_ok=True)
    print(f"WMO {wmo}: downloading from {url} ...")
    urllib.request.urlretrieve(url, out_path)
    size_mb = out_path.stat().st_size / 1e6
    print(f"WMO {wmo}: saved {size_mb:.1f} MB → {out_path}")

print("\nDone.")

WMO 7902194: Rtraj already present — skipping.
WMO 4903784: Rtraj already present — skipping.
WMO 1902682: downloading from https://data-argo.ifremer.fr/dac/coriolis/1902682/1902682_Rtraj.nc ...
WMO 1902682: saved 2.0 MB → ../data/argo_data/argo_1902682/1902682_Rtraj.nc
WMO 7900584: downloading from https://data-argo.ifremer.fr/dac/coriolis/7900584/7900584_Rtraj.nc ...
WMO 7900584: saved 64.2 MB → ../data/argo_data/argo_7900584/7900584_Rtraj.nc
WMO 3902117: downloading from https://data-argo.ifremer.fr/dac/coriolis/3902117/3902117_Rtraj.nc ...
WMO 3902117: saved 6.7 MB → ../data/argo_data/argo_3902117/3902117_Rtraj.nc

Done.


In [8]:
# ── Cell 4: Setup — imports, manifest, bathymetry ─────────────────────────────
import math
from datetime import datetime

import numpy as np
import pandas as pd
import xarray as xr

sys.path.insert(0, str(Path("..") / "src"))
from sim_types import ControlAction, GeoLocation, ProfilerState
from particle_mover import run_until_next_action
from data_loader import (
    load_manifest, load_bathymetry, build_bathymetry_interpolator
)

DATA_DIR      = Path("..") / "data" / "raw"
ARGO_DATA_DIR = Path("..") / "data" / "argo_data"
BATHY_PATH    = DATA_DIR / "D6_2024.nc"

manifest     = load_manifest(DATA_DIR)
bathy_ds     = load_bathymetry(BATHY_PATH)
bathy_interp = build_bathymetry_interpolator(bathy_ds)

# CMEMS time window (re-derived so this cell is self-contained)
t_start = min(datetime.fromisoformat(e["time"][0]) for e in manifest)
t_end   = max(datetime.fromisoformat(e["time"][1]) for e in manifest)

print(f"Manifest: {len(manifest)} tiles | CMEMS window: {t_start.date()} → {t_end.date()}")
print("Bathymetry loaded.")

Manifest: 400 tiles | CMEMS window: 2023-10-01 → 2025-01-31
Bathymetry loaded.


In [9]:
# ── Cell 5: Helper functions ───────────────────────────────────────────────────

DESC_CODE    = 190
PARKING_CODE = 290
ASC_CODE     = 590


def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(math.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = math.sin(dlat / 2) ** 2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlon / 2) ** 2
    return 6371.0 * 2 * math.asin(math.sqrt(a))


def _speed_from_pres_juld(sub_ds):
    """Mean dbar/s slope from a sub-dataset with PRES and JULD."""
    pres = sub_ds.PRES.values.astype(float)
    juld = sub_ds.JULD.values
    dt_s = np.diff(juld) / np.timedelta64(1, "s")
    valid = dt_s != 0
    if not valid.any():
        return None
    return float(np.nanmean(np.diff(pres)[valid] / dt_s[valid]))


def extract_cycles(rtraj_path, bathy_interp):
    """Derive per-cycle metadata from an Argo Rtraj NetCDF file."""
    ds = xr.open_dataset(rtraj_path)
    all_cycles = [int(c) for c in np.unique(ds.CYCLE_NUMBER.values)
                  if not np.isnan(float(c))]
    cycle_numbers = sorted(all_cycles)

    # ── Pass 1: per-cycle positions, timing, speeds, parking depth ────────────
    meta = {}
    for cyc in cycle_numbers:
        mask     = ds.CYCLE_NUMBER.values == cyc
        cyc_ds   = ds.isel(N_MEASUREMENT=mask)
        juld     = cyc_ds.JULD.values
        valid_t  = juld[~np.isnat(juld)]
        if len(valid_t) == 0:
            continue

        lats     = cyc_ds.LATITUDE.values.astype(float)
        lons     = cyc_ds.LONGITUDE.values.astype(float)
        pos_mask = ~np.isnan(lats) & ~np.isnan(lons)
        if not pos_mask.any():
            continue

        codes    = cyc_ds.MEASUREMENT_CODE.values
        desc_ds  = cyc_ds.isel(N_MEASUREMENT=codes == DESC_CODE)
        asc_ds   = cyc_ds.isel(N_MEASUREMENT=codes == ASC_CODE)
        park_ds  = cyc_ds.isel(N_MEASUREMENT=codes == PARKING_CODE)

        desc_t   = desc_ds.JULD.values; desc_t = desc_t[~np.isnat(desc_t)]
        asc_t    = asc_ds.JULD.values;  asc_t  = asc_t[~np.isnat(asc_t)]

        park_p   = park_ds.PRES.values.astype(float)
        park_p   = park_p[~np.isnan(park_p)]

        start_lat = float(lats[pos_mask][0])
        start_lon = float(lons[pos_mask][0])
        last_lat  = float(lats[pos_mask][-1])
        last_lon  = float(lons[pos_mask][-1])

        meta[cyc] = dict(
            cycle=cyc,
            start_lat=start_lat,
            start_lon=start_lon,
            last_lat=last_lat,
            last_lon=last_lon,
            start_time=str(valid_t[0]),
            end_time=str(valid_t[-1]),
            first_desc_juld=desc_t[0]  if len(desc_t) > 0 else valid_t[0],
            last_asc_juld=asc_t[-1]    if len(asc_t)  > 0 else valid_t[-1],
            descent_speed_dbar_s=_speed_from_pres_juld(desc_ds)
                                 if len(desc_ds.N_MEASUREMENT) >= 2 else None,
            ascent_speed_dbar_s=_speed_from_pres_juld(asc_ds)
                                if len(asc_ds.N_MEASUREMENT) >= 2 else None,
            avg_bottom_depth_dbar=float(np.mean(park_p)) if len(park_p) > 0 else None,
            bathy_depth_m=float(bathy_interp(start_lat, start_lon)),
        )

    # ── Pass 2: surface drift → parked_on_bottom ──────────────────────────────
    for i, cyc in enumerate(cycle_numbers[:-1]):
        nxt = cycle_numbers[i + 1]
        if cyc not in meta or nxt not in meta:
            continue
        m, nm = meta[cyc], meta[nxt]
        surf_min = float(
            (nm["first_desc_juld"] - m["last_asc_juld"]) / np.timedelta64(1, "m")
        )
        dlat_m = (nm["start_lat"] - m["last_lat"]) * 111320.0
        dlon_m = ((nm["start_lon"] - m["last_lon"])
                  * 111320.0 * math.cos(math.radians(m["last_lat"])))
        drift_km = math.sqrt(dlat_m ** 2 + dlon_m ** 2) / 1000.0
        m["surface_duration_min"] = round(max(surf_min, 0.0), 1)
        m["drift_km"]             = round(drift_km, 3)
        m["parked_on_bottom"]     = drift_km <= 2.0

    # Default for last cycle
    if cycle_numbers and cycle_numbers[-1] in meta:
        last = meta[cycle_numbers[-1]]
        last.setdefault("surface_duration_min", 30.0)
        last.setdefault("drift_km", None)
        last.setdefault("parked_on_bottom", False)

    # ── Fill missing speeds with per-float mean ────────────────────────────────
    desc_speeds = [v["descent_speed_dbar_s"] for v in meta.values()
                   if v["descent_speed_dbar_s"] is not None]
    asc_speeds  = [v["ascent_speed_dbar_s"]  for v in meta.values()
                   if v["ascent_speed_dbar_s"]  is not None]
    mean_desc = float(np.mean(desc_speeds)) if desc_speeds else 0.08
    mean_asc  = float(np.mean(asc_speeds))  if asc_speeds  else 0.08
    for v in meta.values():
        if v["descent_speed_dbar_s"] is None:
            v["descent_speed_dbar_s"] = mean_desc
        if v["ascent_speed_dbar_s"] is None:
            v["ascent_speed_dbar_s"] = mean_asc

    ds.close()
    return [meta[c] for c in cycle_numbers if c in meta]


def action_from_cycle(cyc, next_cyc):
    """Build a ControlAction from a cycle dict and the following cycle dict."""
    t_start = np.datetime64(cyc["start_time"])
    t_next  = np.datetime64(next_cyc["start_time"])
    surf_min = float(cyc.get("surface_duration_min") or 30.0)

    try:
        asc_s = cyc["avg_bottom_depth_dbar"] / abs(cyc["ascent_speed_dbar_s"])
        cycle_hours = float(
            (t_next - t_start
             - np.timedelta64(int(surf_min), "m")
             - np.timedelta64(int(asc_s), "s"))
            / np.timedelta64(1, "h")
        )
    except Exception:
        cycle_hours = float((t_next - t_start) / np.timedelta64(1, "h"))

    if cyc["parked_on_bottom"]:
        park_mode = "park_on_bottom"
    elif cyc["avg_bottom_depth_dbar"] is not None:
        park_mode = "parking_depth"
    else:
        park_mode = "drift_on_surface"

    return ControlAction(
        park_mode=park_mode,
        cycle_hours=max(cycle_hours, 0.5),
        transmission_duration_minutes=surf_min,
        target_depth=cyc["avg_bottom_depth_dbar"],
        descent_speed_ms=abs(cyc["descent_speed_dbar_s"]),
        ascent_speed_ms=abs(cyc["ascent_speed_dbar_s"]),
    )


print("Helpers defined.")

Helpers defined.


In [ ]:
# ── Cell 6: Run reset simulation for every float and cycle ────────────────────
# Reset mode: re-initialise the float at its real observed position at the
# start of every cycle. This isolates the single-cycle drift error.

t_start_np = np.datetime64(t_start)
t_end_np   = np.datetime64(t_end)

all_results = {}   # wmo → list[dict]

for wmo in WMOS_TO_ANALYZE:
    rtraj_path = ARGO_DATA_DIR / f"argo_{wmo}" / f"{wmo}_Rtraj.nc"
    if not rtraj_path.exists():
        print(f"WMO {wmo}: Rtraj not found — skipping (run Cell 3 first).")
        continue

    print(f"\nWMO {wmo}: extracting cycles...")
    cycles = extract_cycles(rtraj_path, bathy_interp)

    # Keep only cycles whose start falls inside the CMEMS time window
    cycles = [c for c in cycles
              if t_start_np <= np.datetime64(c["start_time"]) <= t_end_np]
    print(f"  {len(cycles)} cycles in CMEMS window")

    float_errors = []
    for i, cyc in enumerate(cycles[:-1]):
        next_cyc = cycles[i + 1]
        action   = action_from_cycle(cyc, next_cyc)

        start_dt = pd.Timestamp(cyc["start_time"]).to_pydatetime().replace(tzinfo=None)
        state = ProfilerState(
            time=start_dt,
            location=GeoLocation(lat=cyc["start_lat"], lon=cyc["start_lon"]),
            depth=0.0,
            phase="communicating",
            x=0.0,
            y=0.0,
            bathymetry_depth=bathy_interp(cyc["start_lat"], cyc["start_lon"]),
        )

        try:
            _, end_state = run_until_next_action(
                state, DATA_DIR, manifest, bathy_interp, action,
                dt_vertical_seconds=30,
            )
            error_km = haversine_km(
                end_state.location.lat, end_state.location.lon,
                next_cyc["start_lat"], next_cyc["start_lon"],
            )
            float_errors.append({
                "wmo": wmo,
                "cycle": cyc["cycle"],
                "start_time": cyc["start_time"],
                "error_km": error_km,
                "sim_lat": end_state.location.lat,
                "sim_lon": end_state.location.lon,
                "real_lat": next_cyc["start_lat"],
                "real_lon": next_cyc["start_lon"],
                "park_mode": action.park_mode,
                "cycle_hours": action.cycle_hours,
            })
        except Exception as exc:
            print(f"  Cycle {cyc['cycle']}: FAILED ({exc})")

    all_results[wmo] = float_errors
    errs = [r["error_km"] for r in float_errors]
    if errs:
        print(f"  n={len(errs)} | "
              f"mean={np.mean(errs):.2f} km | "
              f"median={np.median(errs):.2f} km | "
              f"max={np.max(errs):.2f} km")

all_df = pd.DataFrame([r for rows in all_results.values() for r in rows])
print(f"\nTotal: {len(all_df)} cycles across {all_df['wmo'].nunique()} floats.")


WMO 7902194: extracting cycles...


/tmp/ipykernel_86409/1569892001.py:29: FutureWarning: In a future version, xarray will not decode the variable 'CLOCK_OFFSET' into a timedelta64 dtype based on the presence of a timedelta-like 'units' attribute by default. Instead it will rely on the presence of a timedelta64 'dtype' attribute, which is now xarray's default way of encoding timedelta64 values.
To continue decoding into a timedelta64 dtype, either set `decode_timedelta=True` when opening this dataset, or add the attribute `dtype='timedelta64[ns]'` to this variable on disk.
To opt-in to future behavior, set `decode_timedelta=False`.
  ds = xr.open_dataset(rtraj_path)


  122 cycles in CMEMS window


/tmp/ipykernel_86409/2797927156.py:29: UserWarning: Discarding nonzero nanoseconds in conversion.
  start_dt = pd.Timestamp(cyc["start_time"]).to_pydatetime().replace(tzinfo=None)
NaN velocity at lat=55.0825 lon=16.1966 depth=70.3 bathy=78.4 time=2024-06-18 18:26:30 — defaulting to 0 during phase descending
NaN velocity at lat=55.0825 lon=16.1966 depth=71.4 bathy=78.4 time=2024-06-18 18:27:00 — defaulting to 0 during phase descending
NaN velocity at lat=55.0825 lon=16.1966 depth=72.6 bathy=78.4 time=2024-06-18 18:27:30 — defaulting to 0 during phase descending
NaN velocity at lat=55.0825 lon=16.1966 depth=73.7 bathy=78.4 time=2024-06-18 18:28:00 — defaulting to 0 during phase descending
NaN velocity at lat=55.0825 lon=16.1966 depth=74.9 bathy=78.4 time=2024-06-18 18:28:30 — defaulting to 0 during phase descending
NaN velocity at lat=55.0825 lon=16.1966 depth=76.0 bathy=78.4 time=2024-06-18 18:29:00 — defaulting to 0 during phase descending
NaN velocity at lat=55.0825 lon=16.1966 depth=

  Cycle 38: FAILED (No tiles found for lat=54.897 lon=19.343 time=[2024-08-13 23:57:00, 2024-08-16 13:14:12] margin=0.5°. Float may have left the dataset domain.)


...

/tmp/ipykernel_86409/1569892001.py:29: FutureWarning: In a future version, xarray will not decode the variable 'CLOCK_OFFSET' into a timedelta64 dtype based on the presence of a timedelta-like 'units' attribute by default. Instead it will rely on the presence of a timedelta64 'dtype' attribute, which is now xarray's default way of encoding timedelta64 values.
To continue decoding into a timedelta64 dtype, either set `decode_timedelta=True` when opening this dataset, or add the attribute `dtype='timedelta64[ns]'` to this variable on disk.
To opt-in to future behavior, set `decode_timedelta=False`.
  ds = xr.open_dataset(rtraj_path)


  161 cycles in CMEMS window


/tmp/ipykernel_86409/2797927156.py:29: UserWarning: Discarding nonzero nanoseconds in conversion.
  start_dt = pd.Timestamp(cyc["start_time"]).to_pydatetime().replace(tzinfo=None)
NaN velocity at lat=55.0615 lon=14.3242 depth=42.4 bathy=45.3 time=2023-11-07 21:00:30.000003 — defaulting to 0 during phase descending
NaN velocity at lat=55.0615 lon=14.3242 depth=44.0 bathy=45.3 time=2023-11-07 21:01:00.000003 — defaulting to 0 during phase descending
NaN velocity at lat=55.0615 lon=14.3242 depth=45.3 bathy=45.3 time=2023-11-07 20:47:00.000003 — defaulting to 0 during phase ascending
NaN velocity at lat=55.0615 lon=14.3242 depth=43.1 bathy=45.3 time=2023-11-07 20:47:30.000003 — defaulting to 0 during phase ascending
/tmp/ipykernel_86409/2797927156.py:29: UserWarning: Discarding nonzero nanoseconds in conversion.
  start_dt = pd.Timestamp(cyc["start_time"]).to_pydatetime().replace(tzinfo=None)
NaN velocity at lat=55.0628 lon=14.3257 depth=42.4 bathy=45.4 time=2023-11-07 20:35:03.000003 — de

  n=160 | mean=2.49 km | median=0.69 km | max=81.06 km

WMO 1902682: extracting cycles...
  77 cycles in CMEMS window


NaN velocity at lat=55.3770 lon=14.9698 depth=61.8 bathy=64.2 time=2023-10-21 05:00:30 — defaulting to 0 during phase descending
NaN velocity at lat=55.3770 lon=14.9698 depth=63.5 bathy=64.2 time=2023-10-21 05:01:00 — defaulting to 0 during phase descending
NaN velocity at lat=55.3770 lon=14.9698 depth=64.2 bathy=64.2 time=2023-10-23 01:00:27 — defaulting to 0 during phase ascending
NaN velocity at lat=55.3770 lon=14.9698 depth=62.4 bathy=64.2 time=2023-10-23 01:00:57 — defaulting to 0 during phase ascending
Working window for uo is 55% NaN at lat=55.291 lon=13.287 — many velocity lookups will default to 0
Working window for vo is 55% NaN at lat=55.291 lon=13.287 — many velocity lookups will default to 0
NaN velocity at lat=55.2908 lon=13.2788 depth=23.3 bathy=nan time=2023-10-28 16:25:30 — defaulting to 0 during phase descending
NaN velocity at lat=55.2908 lon=13.2788 depth=24.8 bathy=nan time=2023-10-28 16:26:00 — defaulting to 0 during phase descending
NaN velocity at lat=55.2908 lo

In [ ]:
# ── Cell 7: Visualisation ─────────────────────────────────────────────────────
import plotly.graph_objects as go
from plotly.subplots import make_subplots

wmos_done = [w for w in WMOS_TO_ANALYZE if w in all_results and all_results[w]]

# ── 1. Box plot: error distribution per float ─────────────────────────────────
fig1 = go.Figure()
for wmo in wmos_done:
    errs = [r["error_km"] for r in all_results[wmo]]
    fig1.add_trace(go.Box(
        y=errs, name=str(wmo),
        boxpoints="all", jitter=0.3, pointpos=-1.8,
        hovertemplate="Error: %{y:.2f} km<extra>WMO " + str(wmo) + "</extra>",
    ))
fig1.update_layout(
    title="Single-cycle position error by float (reset mode)",
    yaxis_title="Error (km)",
    xaxis_title="Float WMO",
    paper_bgcolor="rgb(15,20,40)", font_color="white", plot_bgcolor="rgb(20,30,50)",
    height=420,
)
fig1.show()

# ── 2. Per-cycle error vs cycle index ─────────────────────────────────────────
fig2 = go.Figure()
for wmo in wmos_done:
    rows = all_results[wmo]
    fig2.add_trace(go.Scatter(
        x=[r["cycle"] for r in rows],
        y=[r["error_km"] for r in rows],
        mode="lines+markers", name=str(wmo),
        hovertemplate="Cycle %{x} — %{y:.2f} km<extra>WMO " + str(wmo) + "</extra>",
    ))
fig2.update_layout(
    title="Per-cycle position error vs cycle number",
    xaxis_title="Cycle number",
    yaxis_title="Error (km)",
    paper_bgcolor="rgb(15,20,40)", font_color="white", plot_bgcolor="rgb(20,30,50)",
    legend=dict(bgcolor="rgba(0,0,0,0.5)"),
    height=420,
)
fig2.show()

# ── 3. Summary table ──────────────────────────────────────────────────────────
rows = []
for wmo in wmos_done:
    errs = [r["error_km"] for r in all_results[wmo]]
    rows.append({
        "WMO": wmo,
        "N cycles": len(errs),
        "Mean (km)": round(np.mean(errs), 2),
        "Median (km)": round(np.median(errs), 2),
        "P90 (km)": round(np.percentile(errs, 90), 2),
        "Max (km)": round(np.max(errs), 2),
    })
print(pd.DataFrame(rows).to_string(index=False))